## pass@k - k번 중 한 번이라도 정답을 생성했는가?

## pass@k란?
* pass@k 는 에이전트가 k번 시도 중 최소 1번 이상 정답 을 생성할 확률을 측정함
    * k가 증가할수록 pass@k 점수는 올라갑니다 (더 많은 시도 = 더 높은 성공 확률)
    * pass@1 = 50%이면, 에이전트가 첫 번째 시도에서 절반의 질문에 정답을 생성한다는 의미
    * 코딩 에이전트에서는 보통 pass@1이 가장 중요 (첫 시도에 맞추는 것이 핵심)
    * 여러 솔루션을 제안하는 경우, 하나만 맞으면 되므로 pass@k가 더 적합
    * pass@k 는 에이전트의 비결정성 (non-determinism)을 고려한 지표

## 평가 프로세스
```
Golden Dataset (LangSmith)
        │
        ▼
┌─────────────────────────────────┐
│   run_agent_to_completion       │  ← 에이전트를 k번 반복 실행
│   (num_repetitions = k)         │
└─────────────────────────────────┘
        │
        ▼
┌─────────────────────────────────┐
│  is_final_answer_correct        │  ← 각 시도마다 정답 여부 평가
│  (evaluator, 개별 run 마다 실행)  │     → 대시보드에 개별 점수 표시
└─────────────────────────────────┘
        │
        ▼
┌─────────────────────────────────┐
│  pass_at_k                      │  ← summary_evaluator로
│  (summary_evaluator, 전체 실행   │     대시보드에 pass@k 표시
│   완료 후 1번 실행)               │
└─────────────────────────────────┘
```

In [3]:
from dotenv import load_dotenv
load_dotenv()
from langsmith import Client
ls_client = Client() 

## Grader 설정

In [4]:
grader_instructions = """You are a teacher grading a quiz.

You will be given a QUESTION, the GROUND TRUTH (correct) RESPONSE, and the STUDENT RESPONSE.

Here is the grade criteria to follow:
(1) Grade the student responses based ONLY on their factual accuracy relative to the ground truth answer.
(2) Ensure that the student response does not contain any conflicting statements.
(3) It is OK if the student response contains more information than the ground truth response, 
    as long as it is factually accurate relative to the ground truth response.

Correctness:
1 means that the student's response meets all of the criteria.
0 means that the student's response does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct."""

In [5]:
from pydantic import BaseModel, Field

class Grade(BaseModel):
    """LLM Judge 채점 결과 - 이진 평가(정답/오답)"""
    correctness: int = Field(description="1 if the student's response aligns with the content of the ground truth, 0 otherwise.")
    reasoning: str = Field(description="A step-by-step explanation of the grading decision.")

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
        model="gemini-3.1-pro-preview",
        temperature=1
    )
grader_llm = llm.with_structured_output(Grade)

## 에이전트 실행 함수 및 Evaluator 정의

In [7]:
from langchain_core.messages import HumanMessage
from marketing_agent import agent  # 마케팅 에이전트 (노트북 16과 다른 에이전트!)

def run_agent_to_completion(inputs):
    """마케팅 에이전트를 실행하는 함수.
    """
    question = inputs["question"]
    
    result = agent.invoke({
        "messages": [HumanMessage(content=question)]
    })

    return result

/Users/a202304035/LLM_Eval_study/marketing_agent.py:31: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_search = TavilySearchResults(max_results=3)


In [8]:
from langchain_core.messages import HumanMessage, SystemMessage

def is_final_answer_correct(inputs, outputs, reference_outputs):
    """각 개별 run에 대해 정답 여부를 평가하는 evaluator.
    이 점수는 대시보드에서 개별 run마다 표시됩니다."""
    question = inputs["question"]
    answer = outputs["messages"][-1].content
    ground_truth_response = reference_outputs["answer"]
    
    grading_input = f"""QUESTION: {question}
ANSWER: {answer}
GROUND TRUTH: {ground_truth_response}"""
    
    grade = grader_llm.invoke([
        SystemMessage(content=grader_instructions),
        HumanMessage(content=grading_input)
    ])

    return grade.correctness

## Summary Evaluator 정의 (pass@k)
* summary_evaluators는 모든 run이 완료된 후 한 번 실행됩니다. 모든 run과 example을 받아서 질문별로 그룹핑한 뒤, pass@k를 계산

> 참고: summary evaluator는 RunTree 객체를 받기 때문에 개별 evaluator의 결과에 직접 접근할 수 없다. 따라서 ls_client.list_feedback()으로 이미 채점된 feedback을 가져와서 계산한다. feedback 동기화 지연을 고려하여 retry 로직이 포함됨

* 결과는 LangSmith 대시보드에서 실험 수준의 요약 지표로 직접 확인

In [9]:
import time
from collections import defaultdict
from langsmith.schemas import Run, Example
from langsmith.evaluation import EvaluationResults, EvaluationResult

def _fetch_feedback_scores(run_ids, retries=5, delay=3):
    """LangSmith API에서 feedback을 가져오되, 모든 run의 feedback이 동기화될 때까지 재시도합니다."""
    for attempt in range(retries):
        feedbacks = list(ls_client.list_feedback(
            run_ids=run_ids,
            feedback_key=["is_final_answer_correct"],
        ))

        score_by_run_id = {fb.run_id: fb.score for fb in feedbacks}

        if len(score_by_run_id) == len(run_ids):
            return score_by_run_id  # 모든 feedback 수신 완료

        print(f"  Waiting for feedback sync... ({len(score_by_run_id)}/{len(run_ids)}, attempt {attempt + 1}/{retries})")
        time.sleep(delay)

    return score_by_run_id  # 최선의 결과 반환

def pass_at_k(runs: list[Run], examples: list[Example]) -> EvaluationResults:
    """Summary evaluator: 질문별로 k번 중 1번이라도 정답이면 pass.

    개별 evaluator(is_final_answer_correct)가 이미 채점한 feedback을
    LangSmith API에서 가져와서 pass@k를 계산합니다.
    """
    run_ids = [run.id for run in runs]
    score_by_run_id = _fetch_feedback_scores(run_ids)

    # 질문(example)별로 correctness 점수 수집
    scores_per_example = defaultdict(list)
    for run, example in zip(runs, examples):
        score = score_by_run_id.get(run.id, 0)
        scores_per_example[example.id].append(score)

    # pass@k: 각 질문에 대해 1번이라도 정답(1)이면 pass
    passed = sum(
        1 for scores in scores_per_example.values()
        if any(s == 1 for s in scores)
    )
    total = len(scores_per_example)

    k = max(len(s) for s in scores_per_example.values())

    return EvaluationResults(
        results=[
            EvaluationResult(
                key=f"pass_at_{k}",
                score=passed / total,
                comment=f"{passed}/{total} examples passed (at least 1 correct in {k} trials)"
            )
        ]
    )

## 마케팅 에이전트 평가 데이터셋 정의
* 아래에서는 마케팅 콘텐츠 생성 평가를 위한 10개의 테스트 케이스를 직접 정의함
* 각 테스트 케이스는 다양한 마케팅 시나리오를 포함

|#	|시나리오|핵심 제약|
|--|-------|-------|
|1|	인스타그램 게시글|	본문 150자 이내, 해시태그 5개|
|2|	이메일 제목 A/B 테스트|	각 제목 50자 이내, 3가지 접근법|
|3|	블로그 도입부|	실제 검색 기반 통계/뉴스, SEO 키워드|
|4|	프레스 릴리즈|	공식 보도자료 형식, 실제 VC 이름|
|5|	랜딩 페이지 히어로|	헤드라인 15자, CTA 10자 이내|
|6|	유튜브 설명글|	타임스탬프, 재료 리스트, 해시태그|
|7|	제품 상세 페이지|	경쟁사 가격 검색, 스펙 설명|
|8|	SMS 마케팅|	90자 이내 (극단적 제한)|
|9|	파트너십 제안 이메일|	인플루언서 트렌드 검색|
|10|SNS 위기 대응 사과문|	200자 이내, 보상 방안 포함|

* answer 필드에는 정답이 아닌 합격 기준이 기록
* LLM Judge가 이 기준에 따라 에이전트의 출력을 평가

## 실험 실행
* evaluators: 각 개별 run마다 실행 → 대시보드에서 run별 정답 여부 확인 가능
* summary_evaluators: 모든 run 완료 후 실행 → 대시보드에서 pass@k 요약 점수 확인 가능

In [10]:
# 코드로 직접 langsmith dataset 을 정의하는 방법
dataset_name = "marketing-agent-n-workers-sweep"

examples = [
    # ---- 1. 인스타그램 게시글 (글자 수 + 해시태그 + 톤) ----
    {
        "question": (
            "친환경 텀블러 브랜드 'GreenSip'의 인스타그램 게시글을 작성해줘.\n"
            "- 타깃: MZ세대 (20~35세)\n"
            "- 톤: 캐주얼하고 위트 있게\n"
            "- 핵심 메시지: 일회용 컵 대신 GreenSip으로 지구를 지키자\n"
            "- 필수 요소: 해시태그 5개, 이모지 3개 이상, CTA 포함\n"
            "- 본문 150자 이내 (해시태그 제외)"
        ),
        "answer": (
            "합격 기준: (1) 본문이 150자 이내, (2) 해시태그 정확히 5개 포함, "
            "(3) 이모지 3개 이상, (4) CTA(행동 유도 문구) 존재, "
            "(5) 캐주얼/위트 있는 톤, (6) 환경 보호 메시지 포함, "
            "(7) 'GreenSip' 브랜드명 언급"
        ),
    },

    # ---- 2. 이메일 제목 A/B 테스트 (극단적 제한) ----
    {
        "question": (
            "온라인 영어 회화 서비스 'TalkNow'의 블랙프라이데이 프로모션 이메일 제목을 작성해줘.\n"
            "- 타깃: 영어 학습에 관심 있는 30~40대 직장인\n"
            "- 할인: 연간 구독 50% 할인\n"
            "- A/B 테스트용으로 서로 다른 접근의 제목 3개를 제안해줘\n"
            "- 각 제목은 50자 이내\n"
            "- 하나는 긴급성 강조, 하나는 혜택 강조, 하나는 호기심 유발"
        ),
        "answer": (
            "합격 기준: (1) 서로 다른 접근법의 제목 3개, (2) 각각 50자 이내, "
            "(3) 하나는 긴급성(한정, 마감 등), 하나는 혜택(50% 할인, 가격 등), "
            "하나는 호기심(질문형, 반전 등), (4) 'TalkNow' 또는 영어 학습 관련 키워드 포함, "
            "(5) 블랙프라이데이/할인 언급"
        ),
    },

    # ---- 3. 블로그 도입부 (검색 필수 + SEO) ----
    {
        "question": (
            "반려동물 건강식 브랜드 'PawFresh'의 블로그 포스트 도입부(첫 2문단)를 작성해줘.\n"
            "- 주제: '반려견 수제 간식, 정말 안전할까?'\n"
            "- 타깃: 반려견을 키우는 30~50대\n"
            "- 톤: 전문적이되 친근하게\n"
            "- 필수 포함: 최근 반려동물 식품 안전 관련 통계 또는 뉴스 1건 (검색해서 찾아줘)\n"
            "- SEO 키워드: '반려견 수제 간식', '강아지 간식 안전' — 자연스럽게 포함\n"
            "- 도입부 끝에 독자의 궁금증을 유발하는 질문 포함"
        ),
        "answer": (
            "합격 기준: (1) 2문단으로 구성, (2) 실제 검색 기반 통계/뉴스 1건 포함 (날조 아닌 것), "
            "(3) SEO 키워드 '반려견 수제 간식'과 '강아지 간식 안전'이 자연스럽게 포함, "
            "(4) 마지막에 독자 궁금증 유발 질문, (5) 전문적이되 친근한 톤, "
            "(6) 'PawFresh' 브랜드 자연스럽게 언급"
        ),
    },

    # ---- 4. 프레스 릴리즈 (팩트 정확성 + 공식 톤) ----
    {
        "question": (
            "AI 스타트업 'NeuraLab'이 시리즈 A 투자 유치를 발표하는 프레스 릴리즈의 첫 3문단을 작성해줘.\n"
            "- 투자금: 150억 원\n"
            "- 리드 투자자: 검색해서 한국의 유명 VC 1곳을 실제로 찾아 사용해줘\n"
            "- 회사 소개: 의료 영상 AI 분석 솔루션\n"
            "- 톤: 공식적, 보도자료 스타일\n"
            "- 필수 구성: 헤드라인 + 날짜/도시 + 본문 3문단\n"
            "- CEO 코멘트 1개 포함 (가상 인물 'CEO 김민수' 사용)"
        ),
        "answer": (
            "합격 기준: (1) 보도자료 형식(헤드라인 + 날짜/도시 + 본문), "
            "(2) 투자금 150억 원 정확히 명시, (3) 실제 존재하는 한국 VC 이름 사용, "
            "(4) CEO 김민수의 직접 인용문 포함, (5) 의료 영상 AI 사업 설명, "
            "(6) 공식적 보도자료 톤"
        ),
    },

    # ---- 5. 랜딩 페이지 히어로 섹션 (다중 요소 + 글자 수) ----
    {
        "question": (
            "온라인 피트니스 플랫폼 'FitAnywhere'의 랜딩 페이지 히어로 섹션 카피를 작성해줘.\n"
            "- 타깃: 헬스장 갈 시간이 없는 20~40대 직장인\n"
            "- 필수 구성 요소:\n"
            "  (a) 헤드라인 (15자 이내)\n"
            "  (b) 서브 헤드라인 (30자 이내)\n"
            "  (c) 혜택 3가지 불릿 포인트\n"
            "  (d) CTA 버튼 텍스트 (10자 이내)\n"
            "- 톤: 에너지 넘치고 동기부여\n"
            "- 핵심 USP: 하루 15분, 장비 없이, 어디서나"
        ),
        "answer": (
            "합격 기준: (1) 헤드라인 15자 이내, (2) 서브 헤드라인 30자 이내, "
            "(3) 혜택 불릿 포인트 정확히 3개, (4) CTA 버튼 텍스트 10자 이내, "
            "(5) '하루 15분', '장비 없이', '어디서나' 중 2개 이상 반영, "
            "(6) 에너지 넘치는 톤, (7) 'FitAnywhere' 브랜드명 포함"
        ),
    },

    # ---- 6. 유튜브 설명글 (타임스탬프 + 구성) ----
    {
        "question": (
            "쿠킹 유튜브 채널 'Chef's Table KR'의 영상 설명글을 작성해줘.\n"
            "- 영상 제목: '10분 만에 완성! 직장인을 위한 초간단 도시락 3종'\n"
            "- 필수 구성:\n"
            "  (a) 영상 소개 (3줄 이내)\n"
            "  (b) 타임스탬프 3개 (각 도시락별)\n"
            "  (c) 재료 리스트 (간략히)\n"
            "  (d) 구독/좋아요 CTA\n"
            "  (e) 관련 해시태그 5개\n"
            "- 톤: 밝고 친근하게"
        ),
        "answer": (
            "합격 기준: (1) 영상 소개 3줄 이내, (2) 타임스탬프 3개(시간 표시 포함), "
            "(3) 재료 리스트 포함, (4) 구독/좋아요 CTA, (5) 해시태그 5개, "
            "(6) 도시락 메뉴 3종이 구체적으로 명시, (7) 밝고 친근한 톤"
        ),
    },

    # ---- 7. 제품 상세 페이지 (검색 필수 + 경쟁 비교) ----
    {
        "question": (
            "무선 이어폰 'SoundPods Pro'의 쿠팡 제품 상세 페이지 상단 카피를 작성해줘.\n"
            "- 스펙: ANC(능동 노이즈 캔슬링), 블루투스 5.3, 배터리 8시간, IPX5 방수\n"
            "- 가격: 79,000원 (출시 기념 30% 할인)\n"
            "- 타깃: 통근하는 20~30대\n"
            "- 필수 구성:\n"
            "  (a) 제품명 + 한 줄 캐치프레이즈\n"
            "  (b) 핵심 스펙 4가지를 소비자 친화적 언어로 설명\n"
            "  (c) 경쟁 제품 대비 차별점 1가지 (검색해서 경쟁사 가격대를 확인해줘)\n"
            "- 톤: 세련되고 신뢰감 있게"
        ),
        "answer": (
            "합격 기준: (1) 캐치프레이즈 포함, (2) 4가지 스펙 모두 소비자 언어로 설명, "
            "(3) 실제 검색 기반 경쟁사 비교 1건, "
            "(4) 가격 79,000원과 30% 할인 언급, (5) 'SoundPods Pro' 브랜드명, "
            "(6) 세련되고 신뢰감 있는 톤"
        ),
    },

    # ---- 8. SMS 마케팅 (극단적 글자 제한) ----
    {
        "question": (
            "카페 프랜차이즈 'BeanBros'의 SMS 마케팅 메시지를 작성해줘.\n"
            "- 프로모션: 이번 주 금요일까지 아메리카노 1+1\n"
            "- 타깃: 기존 멤버십 고객\n"
            "- 전체 메시지 90자 이내 (링크 URL 제외)\n"
            "- 필수 포함: 브랜드명, 프로모션 내용, 마감일, 매장 방문 유도\n"
            "- [무료 수신 거부 080-XXX-XXXX] 문구는 별도이므로 글자 수에 포함하지 마"
        ),
        "answer": (
            "합격 기준: (1) 전체 90자 이내, (2) 'BeanBros' 브랜드명 포함, "
            "(3) 아메리카노 1+1 프로모션 명시, (4) 금요일까지 마감일 언급, "
            "(5) 매장 방문 유도 문구, (6) 간결하고 임팩트 있는 문체"
        ),
    },

    # ---- 9. 파트너십 제안 이메일 (비즈니스 톤 + 검색 필수) ----
    {
        "question": (
            "인플루언서 마케팅 플랫폼 'LinkUp'이 뷰티 브랜드에 보내는 파트너십 제안 이메일 본문을 작성해줘.\n"
            "- 타깃 수신자: 뷰티 브랜드의 마케팅 팀장\n"
            "- LinkUp의 강점: 뷰티 카테고리 인플루언서 5,000명 네트워크, 평균 ROI 3.2배\n"
            "- 제안 내용: 1개월 무료 체험 후 결정\n"
            "- 톤: 프로페셔널하되 강압적이지 않게\n"
            "- 필수: 구체적 수치 2개 이상, 미팅 제안 CTA, 최근 뷰티 인플루언서 마케팅 트렌드 1건 언급 (검색 필요)"
        ),
        "answer": (
            "합격 기준: (1) 이메일 형식(인사 + 본문 + 마무리), (2) 수치 2개 이상(5000명, ROI 3.2배 등), "
            "(3) 1개월 무료 체험 제안, (4) 미팅/콜 제안 CTA, "
            "(5) 실제 검색 기반 뷰티 인플루언서 트렌드 1건, "
            "(6) 프로페셔널하지만 부드러운 톤, (7) 'LinkUp' 브랜드명"
        ),
    },

    # ---- 10. 소셜 미디어 위기 대응 (공감 + 다중 필수 요소) ----
    {
        "question": (
            "배달 앱 'QuickEats'에서 대규모 배달 지연 사태가 발생했어. "
            "공식 SNS에 올릴 사과문을 작성해줘.\n"
            "- 상황: 시스템 장애로 2시간 동안 전국 배달 지연\n"
            "- 톤: 진정성 있는 사과, 책임감 있게, 변명하지 않기\n"
            "- 필수 포함:\n"
            "  (a) 명확한 사과\n"
            "  (b) 문제 원인 간략 설명 (기술적 세부사항은 빼고)\n"
            "  (c) 이미 취한 조치\n"
            "  (d) 피해 고객 보상 방안 (구체적으로)\n"
            "  (e) 재발 방지 약속\n"
            "- 200자 이내"
        ),
        "answer": (
            "합격 기준: (1) 200자 이내, (2) 명확한 사과 문구, (3) 원인 설명(변명 아닌 설명), "
            "(4) 이미 취한 조치 명시, (5) 구체적 보상 방안(쿠폰, 환불 등), "
            "(6) 재발 방지 약속, (7) 진정성 있고 책임감 있는 톤, "
            "(8) 'QuickEats' 브랜드명"
        ),
    },
]

In [11]:
# 데이터셋이 이미 존재하면 재사용, 없으면 새로 생성
try:
    dataset = ls_client.read_dataset(dataset_name=dataset_name)
    print(f"기존 Dataset '{dataset_name}' 사용")
except Exception:
    # 데이터셋 생성 후 각 예제를 inputs/outputs로 등록
    dataset = ls_client.create_dataset(
        dataset_name=dataset_name,
        description="마케팅 에이전트 N-worker sweep 평가용",
    )
    for ex in examples:
        ls_client.create_example(
            dataset_id=dataset.id,
            inputs={"question": ex["question"]},   # 에이전트에 전달될 입력
            outputs={"answer": ex["answer"]},       # 평가 기준 (합격 조건)
        )
    print(f"Dataset '{dataset_name}' 생성 완료 - {len(examples)}개 examples")

Dataset 'marketing-agent-n-workers-sweep' 생성 완료 - 10개 examples


In [ ]:
# pass@k 실험: k=3 (각 질문을 3번 반복 실행)
experiment_result = ls_client.evaluate(
    run_agent_to_completion,
    data=dataset_name,
    evaluators=[is_final_answer_correct],      # 개별 run마다 정답/오답 평가
    summary_evaluators=[pass_at_k],            # 전체 완료 후 pass@k 계산
    experiment_prefix="pass-at-k-3",
    max_concurrency=2,
    num_repetitions=3,                         # k=3
)

/Users/a202304035/LLM_Eval_study/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'pass-at-k-3-35b4a32c' at:
https://smith.langchain.com/o/2f003b67-27f4-48d0-8856-e1db2397e3d4/datasets/8a1e20cd-bf59-44dc-96d3-93e3941cb92c/compare?selectedSessions=86a2672d-79ac-4f92-b8ec-b0cc248b2461




28it [12:57, 35.02s/it]

### k=5로 추가 실험
* k를 늘려서 pass@k가 어떻게 변하는지 비교
    * 이론적으로 k가 클수록 pass@k는 올라감 (더 많은 시도 = 더 높은 성공 확률).

In [ ]:
# pass@k 실험: k=5 (각 질문을 5번 반복 실행)
# k=3 대비 pass@k가 얼마나 올라가는지 비교
experiment_result = ls_client.evaluate(
    run_agent_to_completion,
    data=dataset_name,
    evaluators=[is_final_answer_correct],
    summary_evaluators=[pass_at_k],
    experiment_prefix="pass-at-k-5-trials",
    max_concurrency=2,
    num_repetitions=5,                         # k=5
)

## 해석 가이드
|결과 패턴|	의미|
|-------|----|
|pass@1 높음|	에이전트가 첫 시도에서 대부분 정답 → 안정적|
|pass@1 낮음, pass@5 높음|	에이전트가 정답을 생성할 능력은 있지만 일관성 부족|
|pass@5도 낮음|	근본적인 능력 부족 → 프롬프트/모델/도구 개선 필요|

## pass@k를 언제 사용하면 좋은가?
* 코딩 에이전트: pass@1이 가장 중요 (첫 시도에 맞추는 것이 핵심)
* 연구/탐색 에이전트: 여러 솔루션 중 하나만 맞으면 되므로 pass@k (k>1)가 더 적합
* 능력 평가 (Capability Eval): 에이전트가 이 작업을 "할 수 있는지" 측정할 때 유용